# Install and Import Libraries

In [1]:
!pip -q install -U datasets transformers accelerate scikit-learn

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 9.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 93.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 18.8 MB/s eta 0:00:00:00:0100:01


## Step 1: Explore the Dataset

In [2]:
ds = load_dataset("ucirvine/sms_spam")["train"]
ds = ds.rename_column("sms", "text")
print(ds.column_names)
print(ds.features["label"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

['text', 'label']
ClassLabel(names=['ham', 'spam'])


## Step 2: Create Label Dictionary

In [3]:
label_map = {0: "ham", 1: "spam"}
id_map = {"ham": 0, "spam": 1}
label_map, id_map


({0: 'ham', 1: 'spam'}, {'ham': 0, 'spam': 1})

## Step 3: Tokenize and Preprocess

In [4]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
tokenized = ds.map(
    lambda b: tokenizer(b["text"], padding="max_length", truncation=True, max_length=128),
    batched=True
)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/5574 [00:00<?, ? examples/s]

## Step 4: Split Train & Evaluation Data

In [5]:
tokenized = tokenized.rename_column("label", "labels")
tokenized = tokenized.remove_columns([c for c in tokenized.column_names if c not in ["input_ids","attention_mask","labels"]])
tokenized = tokenized.shuffle(seed=42)

n = len(tokenized)
train_n = min(5000, max(0, n - 1000))
eval_n = min(1000, n - train_n)

train_ds = tokenized.select(range(train_n))
eval_ds = tokenized.select(range(train_n, train_n + eval_n))

train_ds.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
eval_ds.set_format(type="torch", columns=["input_ids","attention_mask","labels"])

len(train_ds), len(eval_ds)


(4574, 1000)

## Step 5: Fine-Tune DistilBERT

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label=label_map,
    label2id=id_map
)

def compute_metrics(eval_pred):
    if hasattr(eval_pred, "predictions"):
        logits = eval_pred.predictions
        labels = eval_pred.label_ids
    else:
        logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1, zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir="./spam_model_runs",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.evaluate()


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-186660967.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.238600
100,0.069400
150,0.062600
200,0.044700
250,0.066400
300,0.029600
350,0.012600
400,0.023100
450,0.017400
500,0.022100


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.02790644019842148,
 'eval_accuracy': 0.993,
 'eval_precision': 0.968,
 'eval_recall': 0.9758064516129032,
 'eval_f1': 0.9718875502008032,
 'eval_runtime': 200.2354,
 'eval_samples_per_second': 4.994,
 'eval_steps_per_second': 0.315,
 'epoch': 2.0}

# Step 6: Save Model

In [ ]:
trainer.save_model("./spam_model")
tokenizer.save_pretrained("./spam_model")


('./spam_model/tokenizer_config.json',
 './spam_model/special_tokens_map.json',
 './spam_model/vocab.txt',
 './spam_model/added_tokens.json',
 './spam_model/tokenizer.json')

# Step 7: Load Model & Make Predictions

In [8]:
tok2 = AutoTokenizer.from_pretrained("./spam_model")
mod2 = AutoModelForSequenceClassification.from_pretrained("./spam_model")
mod2.eval()

def predict_with_label(t):
    x = tok2(t, return_tensors="pt", padding="max_length", truncation=True, max_length=128)
    with torch.no_grad():
        logits = mod2(**x).logits
    pred = int(torch.argmax(logits, dim=-1).item())
    return mod2.config.id2label[pred]

texts = [
    "Congratulations! You've won a free ticket.",
    "Hey, are we meeting tomorrow?",
]
for t in texts:
    print(predict_with_label(t))


ham
ham


In [12]:
tests = [
    "WINNER!! Call 09061701461 to claim your prize now!",
    "URGENT! You have won $1000. Click http://bit.ly/claim",
    "Hey, are we meeting tomorrow?",
    "I'll call you later tonight"
]

for t in tests:
    print(t, "->", predict_with_label(t))


WINNER!! Call 09061701461 to claim your prize now! -> spam
URGENT! You have won $1000. Click http://bit.ly/claim -> spam
Hey, are we meeting tomorrow? -> ham
I'll call you later tonight -> ham
